# CV Document Classification: Qualifications, Skills, Experience

This notebook loads the trained CV classifier model and applies it directly to CV documents (PDFs).

Given a resume/CV PDF, it will:
- Extract raw text from the document
- Clean and split the text into smaller segments
- Classify each segment as **Qualification**, **Skill**, or **Experience**
- Aggregate and display the results by category.


In [1]:
import pdfplumber
import re
import string
import torch
from collections import defaultdict

In [2]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model = AutoModelForSequenceClassification.from_pretrained("./cv_classifier_model")
tokenizer = AutoTokenizer.from_pretrained("./cv_classifier_model")


C:\Users\dehem\anaconda3\envs\projectdata\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import joblib
le = joblib.load("label_encoder.pkl")
id2label = dict(enumerate(le.classes_))

In [4]:
CONFIDENCE_THRESHOLD = 0.60  # adjust if needed

SECTION_KEYWORDS = {
    "Experience": [
        "work experience",
        "professional experience",
        "employment history",
        "experience"
    ],
    "Skill": [
        "skills",
        "technical skills",
        "technologies",
        "core competencies"
    ],
    "Qualification": [
        "education",
        "academic background",
        "qualification"
    ]
}

DEGREE_KEYWORDS = [
    "bsc", "msc", "phd",
    "bachelor", "master",
    "university", "college"
]


In [6]:
def extract_text_from_pdf(pdf_path: str) -> str:
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text() or ""
            text += page_text + "\n"
    return text


In [7]:
def clean_line(text: str) -> str:
    text = text.strip()
    if not text:
        return ""
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    return text.strip()

In [8]:
def split_into_lines(text: str):
    raw_lines = text.split("\n")
    merged_lines = []
    buffer = ""

    for line in raw_lines:
        line = line.strip()
        if not line:
            continue

        # If line looks like continuation (starts lowercase)
        if buffer and line and line[0].islower():
            buffer += " " + line
        else:
            if buffer:
                merged_lines.append(buffer.strip())
            buffer = line

    if buffer:
        merged_lines.append(buffer.strip())

    return merged_lines

In [9]:
def detect_section_header(line: str):
    line_lower = line.lower()
    for section, keywords in SECTION_KEYWORDS.items():
        for keyword in keywords:
            if keyword in line_lower:
                return section
    return None

In [10]:
def rule_based_classification(line: str):
    line_lower = line.lower()

    # If contains year → Experience
    if re.search(r"\b(19|20)\d{2}\b", line_lower):
        return "Experience"

    # Degree keywords → Qualification
    if any(word in line_lower for word in DEGREE_KEYWORDS):
        return "Qualification"

    # Comma-separated tech stack → Skill
    if "," in line and len(line.split(",")) >= 3:
        return "Skill"

    return None

In [11]:
@torch.inference_mode()
def classify_line(text: str) -> str:
    if not text.strip():
        return "Other"

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=-1)
    confidence, pred_id = torch.max(probs, dim=-1)

    confidence = confidence.item()
    pred_id = pred_id.item()

    if confidence < CONFIDENCE_THRESHOLD:
        return "Other"

    return id2label.get(pred_id, "Other")


In [12]:
def looks_like_skill_line(line):
    tech_keywords = []
    nlp_test = spacy.load("skill_ner_model")
    doc = nlp_test("pandas")
    for ent in doc.ents:
        tech_keywords.append(ent.text)
    
    count = count(tech_keywords)
    
    return count >= 2

In [13]:
from collections import defaultdict

def classify_document(pdf_path: str):
    raw_text = extract_text_from_pdf(pdf_path)
    lines = split_into_lines(raw_text)

    results = defaultdict(list)
    current_section = None

    for line in lines:
        cleaned = clean_line(line)
        if not cleaned:
            continue

        # 1️⃣ Detect section header
        detected_section = detect_section_header(line)
        if detected_section:
            current_section = detected_section
            continue

        # 2️⃣ Rule-based override first
        rule_label = rule_based_classification(line)
        if rule_label:
            results[rule_label].append(line.strip())
            continue

        # 3️⃣ If inside a section → trust structure
        if current_section:
            results[current_section].append(line.strip())
            continue

        # 4️⃣ ML fallback (only if no structure + no rule)
        label = classify_line(cleaned)
        results[label].append(line.strip())

    return results

In [14]:
def print_results(results):
    for section, lines in results.items():
        print(f"\n=== {section} ({len(lines)} lines) ===")
        for line in lines:
            print("-", line)

In [17]:
results = classify_document("resume_pdfs/resume_00005_Python_Developer.pdf")
print_results(results)


=== Skill (32 lines) ===
- Category: Python_Developer
- Bootstrap, JavaScript, jQuery, jQuery UI, jQuery Mobile to make better Single Page Application SPA working on Node.JS Server along with this worked on react.JS as well.
- AWS kinesis Firehouse, AWS SQS, AWS cloud Watch, AWS SNS, and S3 buckets, EC2 Instances, AWS
- UNIX/Linux, Subversion(SVN), Jenkins, VMware, Ansible, Chef, Puppet, Vagrant, Docker, Tomcat,
- Kubernetes.
- Flask and Swagger. Modeled the application to support bi-directional communication between server and client using Akka HTTP. Designed front-end presentation logic using HTML, CSS, JavaScript, Ajax, AngularJS and
- Conventions Enforcement, Using Python; Stored Procedures created a server job that enforces naming conventions for all SQL objects. Cr eating a more constant SQL environment. Developed Python scripts to monitor health of Mongo databases and perform ad-hoc backups using Mongo dump and Mongo restore. Created test automation framework from scratch using

In [18]:
results['Qualification']

['Sr. Full Stack Python Developer Sr. Full Stack Python Developer Sr. Full Stack Python Developer - Autodesk',
 "Bachelor's"]